[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/lab05_predicting_outcomes.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Lab 5 — <your last name>` (click the title at the top left).

**To submit:** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Lab 5 — Predicting outcomes I: regression, then trees
*Week 5. Baseline: sections 1–5. Powerups: section 6.*

Which hospitals end up in Medicare's penalty range for heart-failure readmissions, and can public
hospital characteristics predict it? You build a logistic regression first, because it is the floor every
fancier model has to beat, then a shallow tree, and you police **leakage** the whole way.

**Unplugged lab.** The baseline (sections 1–5) is done **without an AI assistant**. Logistic regression is one of
the two skills every analyst should be able to do by hand. Powerups may use an assistant as usual.

**How labs work.** Save a copy in Drive, work in your copy, share it back with the instructor
when you submit. Every lab has a **baseline** (everyone finishes it in the session) and
**powerups** (optional depth). Cells marked `# TODO` are yours to fill in.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


In [ ]:
# --- Week 5 packages: scikit-learn and statsmodels ---------------------------------
# Not every base image has them (Hex does not). Installs once per kernel session.
import importlib.util, shutil, subprocess, sys

_need = [pkg for pkg, mod in [("scikit-learn", "sklearn"), ("statsmodels", "statsmodels")]
         if importlib.util.find_spec(mod) is None]
if _need:
    if shutil.which("uv"):  # Hex: the kernel's virtualenv has no pip module, but uv is on the PATH
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_need], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=True)
    importlib.invalidate_caches()
print("week 5 packages ready")


## 1. The target: is a hospital in the penalty range?
From the HRRP file (week 2), keep heart failure (`READM-30-HF-HRRP`) and the rows with a numeric
**Excess Readmission Ratio**. A hospital with a ratio above 1 is in the range where Medicare reduces payments,
and that is our binary target, `penalized`.

**Baseline:** report how many hospitals have a target and what share is penalized. Then look at the
alternative target, `Compared to National` for `READM_30_HF` in *Unplanned Hospital Visits*, and say in one
sentence why we did not use it. *(Hint: count its values before you think about modeling it.)*

In [ ]:
hrrp = load_cms("hrrp")
hf = hrrp[hrrp["Measure Name"].eq("READM-30-HF-HRRP")].copy()
hf["excess_ratio"] = to_num(hf["Excess Readmission Ratio"])
hf = hf.dropna(subset=["excess_ratio"])
hf["penalized"] = (hf["excess_ratio"] > 1).astype(int)
# TODO: number of hospitals, share penalized; then the balance of "Compared to National" for READM_30_HF.


**Why not `Compared to National`?** *(edit this cell)*

## 2. Predictors, and the leakage check
A predictor **leaks** when it contains the answer: it is computed from the target, or measured in a way that
would not be available when you would actually make the prediction. A leaky model scores beautifully and
is useless.

The cell below joins the target to *Hospital General Information*. **Baseline:** for every column in
`CANDIDATES`, decide **keep** or **drop** and give a one-line reason in the table. At least four of them leak,
some obviously and some not. Read the column names carefully, and look up what the star rating is built from.

In [ ]:
hgi = load_cms("hospital_general_info")
df = hf[["Facility ID", "penalized", "Number of Discharges", "Number of Readmissions",
         "Predicted Readmission Rate", "Expected Readmission Rate"]].merge(hgi, on="Facility ID", how="inner")

CANDIDATES = [
    "Hospital Type", "Hospital Ownership", "Emergency Services", "State",
    "Number of Discharges", "Number of Readmissions",
    "Predicted Readmission Rate", "Expected Readmission Rate",
    "Hospital overall rating",
    "Count of READM Measures Better", "Count of READM Measures Worse",
    "Count of MORT Measures Worse", "Count of Facility Safety Measures",
]
print(df.shape)
df[CANDIDATES].head()


**Keep or drop (edit this cell):**

| Column | Keep / drop | Reason |
|---|---|---|
| Hospital Type | | |
| Hospital Ownership | | |
| Emergency Services | | |
| State | | |
| Number of Discharges | | |
| Number of Readmissions | | |
| Predicted Readmission Rate | | |
| Expected Readmission Rate | | |
| Hospital overall rating | | |
| Count of READM Measures Better | | |
| Count of READM Measures Worse | | |
| Count of MORT Measures Worse | | |
| Count of Facility Safety Measures | | |

## 3. Logistic regression as the floor
Fit a logistic regression of `penalized` on the predictors you kept, with **statsmodels**, so you get
coefficients and standard errors rather than just predictions.

**Baseline:**
1. Convert the numeric columns you kept with `to_num`, drop rows with missing values, and report how many
   hospitals that leaves. (Who got dropped? Is that a problem?)
2. Fit the model and print the summary.
3. Turn the coefficients into **odds ratios** (`np.exp(fit.params)`) and interpret two of them in plain English,
   for example, "a for-profit hospital has *x* times the odds of being penalized as a voluntary non-profit, holding
   the rest constant." One numeric, one categorical.

*Tip:* statsmodels compares every category with a baseline, which by default is the first one alphabetically.
Count each category first, and pick a large, meaningful baseline with
`C(hospital_ownership, Treatment('Voluntary non-profit - Private'))`.

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

# Formulas cannot handle spaces in names; snake_case everything first.
m = df.rename(columns=lambda c: c.strip().lower().replace(" ", "_").replace("/", "_"))
# TODO: to_num your numeric columns, dropna, then for example:
# fit = smf.logit("penalized ~ C(hospital_ownership) + C(emergency_services) + ...", data=m).fit()
# fit.summary(); np.exp(fit.params)


**Two odds ratios, in plain English:** *(edit this cell)*

## 4. Does it beat guessing?
In-sample fit flatters every model. Hold out 20% of hospitals (`stratify` on the target, `random_state=42`),
refit on the other 80% with scikit-learn's `LogisticRegression`, and score the hold-out.

**Baseline:** report hold-out **accuracy** next to the accuracy of always guessing the majority class, and the
**AUROC**. One sentence: is the model useful? (Week 6 goes much deeper on evaluation.)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# TODO: NUMERIC = [...]; CATEGORICAL = [...]  (the columns you kept, snake_case names)
# TODO: split, build Pipeline([("prep", ColumnTransformer(...)), ("clf", LogisticRegression(max_iter=1000))]),
#       fit on train, score the hold-out: accuracy, majority-class accuracy, AUROC.


## 5. When trees help
A tree can find interactions and thresholds ("small rural hospitals without an emergency department") that a
logistic model only finds if you write them in.

**Baseline:** fit a `DecisionTreeClassifier(max_depth=3)` on the same split and predictors, score it the same way,
and draw it with `sklearn.tree.plot_tree`. Two sentences: did the tree beat the logistic floor, and what does its
first split tell you?

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# TODO: same preprocessing, DecisionTreeClassifier(max_depth=3, random_state=42); score; plot_tree(...)


**Tree vs floor:** *(edit this cell)*

## 6. Powerups
**(a) Watch leakage happen.** Add `Predicted Readmission Rate` and `Expected Readmission Rate` back in and refit.
Report the new AUROC and explain, using the week 2 definition of the excess readmission ratio, why the number
is meaningless.

**(b) XGBoost with SHAP.** Install `xgboost` and `shap` the same way as the week 5 packages cell, fit an
`XGBClassifier` on your kept predictors, and make a SHAP summary plot. Does it agree with your odds ratios?

**(c) Place.** Bring in the county `ACCESS2` (uninsured) and `DIABETES` measures from lab 4 with your fixed
county join, and refit. Did community context help more than anything about the hospital itself?

In [ ]:
# TODO (powerups)
